In [1]:
%load_ext autoreload
%autoreload 2

# Imports

In [2]:
from pathlib import Path
import shutil
import os

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import open3d as o3d
from torch import Tensor, nn
import MinkowskiEngine as ME
import faiss
from tqdm import tqdm


from opr.models.place_recognition import MinkLoc3D, MinkLoc3Dv2
from opr.pipelines.place_recognition import PlaceRecognitionPipeline

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


/usr/local/lib/python3.10/dist-packages/MinkowskiEngine-0.5.4-py3.10-linux-x86_64.egg/MinkowskiEngine/__init__.py:36: UserWarning: The environment variable `OMP_NUM_THREADS` not set. MinkowskiEngine will automatically set `OMP_NUM_THREADS=16`. If you want to set `OMP_NUM_THREADS` manually, please export it on the command line before running a python script. e.g. `export OMP_NUM_THREADS=12; python your_program.py`. It is recommended to set it below 24.
  warnings.warn(
2025-08-12 23:04:23.484 | WARNING  | opr.models.place_recognition.pointmamba:<module>:16 - The 'pointmamba' package is not installed. Please install it manually if neccessary.


# Constants

In [3]:
REPO_ROOT = Path("/home/docker_mmpr/multimodal-place-recognition")
DATASETS_ROOT = Path("/home/docker_mmpr/Datasets/")

AIRI_DATA_DIR = DATASETS_ROOT / "2024-12-14-AIRI-dataset" / "processed-data" / "slam" / "slam"
assert AIRI_DATA_DIR.exists(), f"Data directory {AIRI_DATA_DIR} does not exist."


# DataReader definition

Соберем словари вида (индекс скана, путь к скану).

In [4]:
maps = [f"AIRI_slam_{i}" for i in range(1, 4)]

map_dicts = []
for i, map in enumerate(maps):
    lidar_path = AIRI_DATA_DIR / map / "keyframe_map" / "scans"

    filenames = sorted(os.listdir(lidar_path))
    idx_to_path = dict()
    for filename in filenames:
        scan_full_path = lidar_path / filename 

        idx_to_path[str(int(filename.split('.')[0]))] = scan_full_path

    map_dicts.append(idx_to_path)

In [5]:
class DataReader:
    def __init__(
            self, 
            csv_file: str | Path, 
            lidar_scans_dirs: list[dict], 
            pointcloud_quantization_size: float = 0.1,
            rename_columns: dict | None = None
        ) -> None:
        """Initialize DataReader for pose-timestamped point cloud data.

        Args:
            csv_file (str | Path): Path to the CSV file containing pose and timestamp data.
            lidar_scans_dir (list[dict]): List of dictionary of (index, path_to_scan) format.
            pointcloud_quantization_size (float): Size for quantizing the point cloud coordinates.
                Default is 0.1.
        Raises:
            FileNotFoundError: If the CSV file or lidar scans directory does not exist.
        """
        csv_file = Path(csv_file)
        if not csv_file.exists():
            raise FileNotFoundError(f"CSV file {csv_file} does not exist.")

        self.df = self.read_csv(csv_file)
        if rename_columns is not None:
            self.df = self.df.rename(columns=rename_columns)

        self.lidar_scans_dirs = lidar_scans_dirs

        self._pointcloud_quantization_size = pointcloud_quantization_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int) -> dict[str, Tensor]:
        """Get the pose and point cloud data for a given index.

        Args:
            idx (int): Index of the data point to retrieve.
        Returns:
            dict: A dictionary containing:
                - pose (Tensor): The pose as a 7-element tensor [x, y, z, qx, qy, qz, qw].
                - pointcloud_lidar_coords (Tensor): The coordinates of the point cloud as an Nx3 tensor.
                - pointcloud_lidar_feats (Tensor): The features of the point cloud as an Nx1 tensor (intensity or ones).
        Raises:
            IndexError: If the index is out of range.
            ValueError: If the scan file is empty or has an unexpected format.
        """
        if idx < 0 or idx >= len(self.df):
            raise IndexError("Index out of range.")

        scan_idx, map_idx = self.df["lidar_timestamp"].iloc[idx].split("m")
        map_idx = int(map_idx) - 1 # DataFrame is 1-indexed

        pose = self.df[["x", "y", "z", "qx", "qy", "qz", "qw"]].iloc[idx].to_numpy()
        
        try:
            scan_filepath = self.lidar_scans_dirs[map_idx][scan_idx]
        except Exception:
            print(map_idx, scan_idx)
        
        pc_coords, pc_feats = self.read_scan(scan_filepath)

        output_dict = {
            "pose": Tensor(pose),
            "pointcloud_lidar_coords": Tensor(pc_coords),
            "pointcloud_lidar_feats": Tensor(pc_feats)
        }

        return output_dict

    def collate_fn(self, batch: list[dict[str, Tensor]]) -> dict[str, Tensor]:
        """Collate function to combine a batch of data points into a single dictionary.
        Args:
            batch (list[dict[str, Tensor]]): A list of dictionaries containing pose and point cloud data.
        Returns:
            dict: A dictionary containing:
                - poses (Tensor): Poses as an Nx7 tensor.
                - pointclouds_lidar_coords (Tensor): Point cloud coordinates as an Nx3 tensor.
                - pointclouds_lidar_feats (Tensor): Point cloud features as an Nx1 tensor.
        """
        poses = torch.stack([item['pose'] for item in batch])

        coords_list = [e["pointcloud_lidar_coords"] for e in batch]
        feats_list = [e["pointcloud_lidar_feats"] for e in batch]
        quantized_coords_list = []
        quantized_feats_list = []
        for coords, feats in zip(coords_list, feats_list):
            quantized_coords, quantized_feats = ME.utils.sparse_quantize(
                coordinates=coords,
                features=feats,
                quantization_size=self._pointcloud_quantization_size,
            )
            quantized_coords_list.append(quantized_coords)
            quantized_feats_list.append(quantized_feats)

        return {
            "poses": poses,
            "pointclouds_lidar_coords": ME.utils.batched_coordinates(quantized_coords_list),
            "pointclouds_lidar_feats": torch.cat(quantized_feats_list)
        }

    def read_scan(self, scan_filepath: str | Path) -> tuple[np.ndarray, np.ndarray]:
        """Read a point cloud scan from a file.
        Args:
            scan_filepath (str | Path): Path to the point cloud file.
        Returns:
            tuple: A tuple containing:
                - coordinates (np.ndarray): The coordinates of the point cloud as an Nx3 array.
                - features (np.ndarray): The features of the point cloud as an Nx1 array (intensity or ones).
        Raises:
            ValueError: If the scan file is empty or has an unexpected format.
        """
        scan = o3d.io.read_point_cloud(str(scan_filepath))
        if not scan.has_points():
            raise ValueError(f"Scan file {scan_filepath} is empty or invalid.")
        # Convert to numpy array for easier manipulation
        scan = np.asarray(scan.points)
        coordinates = scan[:, :3]  # Get the first three columns (x, y, z)
        if scan.shape[1] == 3:
            features = np.ones((coordinates.shape[0], 1))
        elif scan.shape[1] == 4:
            features = scan[:, 3:4]  # Get the fourth column (intensity)
        else:
            raise ValueError(f"Unexpected scan format with shape {scan.shape}. Expected 3 or 4 columns.")
        return coordinates, features

    def read_csv(self, filepath: str | Path) -> pd.DataFrame:
        """Read a CSV file containing pose and timestamp data.
        Args:
            filepath (str | Path): Path to the CSV file.
        Returns:
            pd.DataFrame: A DataFrame containing the pose and timestamp data.
        Raises:
            FileNotFoundError: If the CSV file does not exist.
        """
        dtype_mapping = {
            'pose_timestamp': np.int64,
            'lidar_timestamp': np.int64,
            'x': np.float64,
            'y': np.float64,
            'z': np.float64,
            'qx': np.float64,
            'qy': np.float64,
            'qz': np.float64,
            'qw': np.float64,
        }
        df = pd.read_csv(filepath, dtype=dtype_mapping)
        return df

# Init datareaders

In [6]:
PC_QUANTIZATION_SIZE = 0.05

# Database: batched processing for efficient descriptor extraction
database_reader = DataReader(
    csv_file=AIRI_DATA_DIR / "db_lidar_poses_full.csv",
    lidar_scans_dirs=map_dicts,
    pointcloud_quantization_size=PC_QUANTIZATION_SIZE,
    rename_columns={"px": "x", "py": "y", "pz": "z", "lidar_ts": "lidar_timestamp"}
)

database_dl = torch.utils.data.DataLoader(
    database_reader,
    batch_size=16,
    shuffle=False,
    collate_fn=database_reader.collate_fn,
    num_workers=4,
    pin_memory=True,
    drop_last=False,
)

# Query: single sample processing (no batching needed)
query_reader = DataReader(
    csv_file=AIRI_DATA_DIR / "query_lidar_poses_full.csv",
    lidar_scans_dirs=map_dicts,
    pointcloud_quantization_size=PC_QUANTIZATION_SIZE,  # Must match database for consistency
    rename_columns={"px": "x", "py": "y", "pz": "z", "lidar_ts": "lidar_timestamp"}
)

# Init model

In [7]:
weights = torch.load(REPO_ROOT / "data" / "checkpoints" / "minkloc3dv2_baseline.pth")
# weights = torch.load(REPO_ROOT / "data" / "checkpoints" / "minkloc3d_nclt.pth")
# weights = torch.load(REPO_ROOT / "data" / "checkpoints" / "minkloc3dv2_nclt.pth")

model = MinkLoc3Dv2()
# model = MinkLoc3D()
model.load_state_dict(weights, strict=False)
model.eval()
if torch.cuda.is_available():
    model = model.cuda()
else:
    print("CUDA is not available, running on CPU.")

# Build Faiss index

In [8]:
# Extract descriptors from all database point clouds
descriptors_list = []
with torch.no_grad():
    for batch in tqdm(database_dl):
        batch = {k: v.to("cuda") for k, v in batch.items()}
        descriptors = model(batch)["final_descriptor"]
        descriptors_list.append(descriptors)
descriptors = torch.cat(descriptors_list, dim=0)
print(f"Descriptors shape: {descriptors.shape}")

# Create L2 distance FAISS index for nearest neighbor search
faiss_index = faiss.IndexFlatL2(descriptors.shape[1])
faiss_index.add(descriptors.cpu().numpy())
faiss.write_index(
    faiss_index,
    str(REPO_ROOT / "data" / "2024-12-14-AIRI-dataset" / "processed-data" / "slam" / "slam" / "index.faiss")
)

# Copy pose data as track.csv (required by PlaceRecognitionPipeline)
shutil.copy(
    AIRI_DATA_DIR / "db_lidar_poses_full.csv",
    REPO_ROOT / "data" / "2024-12-14-AIRI-dataset" / "processed-data" / "slam" / "slam" / "track.csv"
)

  0%|          | 0/30 [00:00<?, ?it/s]

100%|██████████| 30/30 [00:01<00:00, 24.21it/s]

Descriptors shape: torch.Size([468, 256])


PosixPath('/home/docker_mmpr/multimodal-place-recognition/data/2024-12-14-AIRI-dataset/processed-data/slam/slam/track.csv')

In [9]:
from copy import deepcopy

class TopKPRPipeline(PlaceRecognitionPipeline):
    def infer_top_k(self, input_data: dict[str, torch.Tensor], top_k: int = 5) -> dict[str, np.ndarray]:
        input_data = self._preprocess_input(input_data)
        output = {}
        with torch.no_grad():
            descriptor = self.model(input_data)["final_descriptor"].cpu().numpy().reshape(1, -1)

        _, predictions = self.database_index.search(descriptor, k=top_k)
        pred_ids = deepcopy(predictions[0]) #
        pred_poses = self.database_df.iloc[pred_ids][['px', 'py', 'pz', 'qx', 'qy', 'qz', 'qw']].to_numpy(dtype=float)
        output["idx"] = pred_ids
        output["pose"] = pred_poses
        output["descriptor"] = descriptor[0]
        return output

In [10]:
pipeline = TopKPRPipeline(
    database_dir=REPO_ROOT / "data" / "2024-12-14-AIRI-dataset" / "processed-data" / "slam" / "slam" ,
    model=model,
    device="cuda",
    pointcloud_quantization_size=PC_QUANTIZATION_SIZE,
)
# Evaluate place recognition accuracy by comparing retrieved vs ground truth poses
translation_errors = []
rotation_errors = []  # angle in radians

top_k_candidates = 5
distance_threshold = 5.0  # meters

recalls = [0] * top_k_candidates

for q_idx, query in tqdm(enumerate(query_reader), total=len(query_reader)):
    query = {k: v.to("cuda") for k, v in query.items()}
    results = pipeline.infer_top_k(query, top_k=top_k_candidates)
    db_idx = results["idx"]  # Retrieved database indices, now it is a list
    q_pose = query["pose"].cpu().numpy()  # Ground truth query pose

    q_loc, q_rot = q_pose[:3], q_pose[3:]

    match_at_k = [False] * top_k_candidates

    best_translation_error, best_rotation_error = float('inf'), float('inf')
    for rank, db_pose in enumerate(results["pose"]): # Retrieved database poses, now it is a list
        db_loc, db_rot = db_pose[:3], db_pose[3:]
        translation_error = np.linalg.norm(q_loc - db_loc)
        rotation_error = 2 * np.arccos(np.abs(np.dot(q_rot, db_rot)))

        best_translation_error = min(best_translation_error, translation_error)
        best_rotation_error = min(best_rotation_error, rotation_error)

        if translation_error < distance_threshold:
            match_at_k[rank:] = [True] * (len(match_at_k) - rank)
            break

    translation_errors.append(best_translation_error)
    rotation_errors.append(best_rotation_error)

    for k in range(top_k_candidates):
        recalls[k] += match_at_k[k]

translation_errors = np.array(translation_errors)
rotation_errors = np.array(rotation_errors)

print(f"Mean translation error: {np.mean(translation_errors):.2f} m")
print(f"Mean rotation error: {np.rad2deg(np.mean(rotation_errors)):.2f} degrees")

print(f"Median translation error: {np.median(translation_errors):.2f} m")
print(f"Median rotation error: {np.rad2deg(np.median(rotation_errors)):.2f} degrees")

total = len(query_reader)
for k in range(top_k_candidates):
    recall_at_k = recalls[k] / total
    print(f"Recall@{k + 1} at {distance_threshold:.1f} m: {recall_at_k:.2%}")

100%|██████████| 485/485 [00:07<00:00, 63.85it/s]

Mean translation error: 4.36 m
Mean rotation error: 130.70 degrees
Median translation error: 2.45 m
Median rotation error: 171.08 degrees
Recall@1 at 5.0 m: 56.29%
Recall@2 at 5.0 m: 65.15%
Recall@3 at 5.0 m: 69.90%
Recall@4 at 5.0 m: 72.99%
Recall@5 at 5.0 m: 76.08%


**Результаты для MinkLoc3Dv1(NCLT):**

Mean translation error: 3.61 m
Mean rotation error: 133.81 degrees
Median translation error: 2.16 m
Median rotation error: 165.53 degrees
Recall@1 at 5.0 m: 58.35%
Recall@2 at 5.0 m: 69.69%
Recall@3 at 5.0 m: 73.20%
Recall@4 at 5.0 m: 75.26%
Recall@5 at 5.0 m: 79.18%

**Результаты для MinkLoc3Dv2(NCLT):**

Mean translation error: 5.08 m
Mean rotation error: 125.36 degrees
Median translation error: 3.00 m
Median rotation error: 162.60 degrees
Recall@1 at 5.0 m: 38.14%
Recall@2 at 5.0 m: 52.37%
Recall@3 at 5.0 m: 59.38%
Recall@4 at 5.0 m: 64.33%
Recall@5 at 5.0 m: 68.25%

**Результаты для MinkLoc3Dv2(Oxford):**

Mean translation error: 4.36 m
Mean rotation error: 130.70 degrees
Median translation error: 2.45 m
Median rotation error: 171.08 degrees
Recall@1 at 5.0 m: 56.29%
Recall@2 at 5.0 m: 65.15%
Recall@3 at 5.0 m: 69.90%
Recall@4 at 5.0 m: 72.99%
Recall@5 at 5.0 m: 76.08%